In [1]:
import requests
import io
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score


In [3]:
FLOW_LOGS_API_ENDPOINT = (
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
)

HOST_TELEMETRY_API_ENDPOINT = (
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
)

In [4]:
def fetch_api_data(url: str) -> pd.DataFrame:
    """
    Fetch CSV data from a REST API endpoint.

    Steps:
    1. Send HTTP GET request.
    2. Check HTTP status.
    3. Convert response text into an in-memory file.
    4. Read CSV into pandas DataFrame.
    5. Return DataFrame.
    """
    response=requests.get(url)
    response.raise_for_status()
    csv_data=io.StringIO(response.text)
    df=pd.read_csv(csv_data)
    return df

In [5]:
flow_logs=fetch_api_data(FLOW_LOGS_API_ENDPOINT)
host_telemetry = fetch_api_data(HOST_TELEMETRY_API_ENDPOINT)


In [6]:
print(flow_logs.head())

  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
3  Adelie  Torgersen             NaN            NaN                NaN   
4  Adelie  Torgersen            36.7           19.3              193.0   

   body_mass_g     sex  
0       3750.0    MALE  
1       3800.0  FEMALE  
2       3250.0  FEMALE  
3          NaN     NaN  
4       3450.0  FEMALE  


In [7]:
flow_logs=flow_logs.dropna().copy() #avoid settingwithcopywarning

In [8]:
print(flow_logs.head())

  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
4  Adelie  Torgersen            36.7           19.3              193.0   
5  Adelie  Torgersen            39.3           20.6              190.0   

   body_mass_g     sex  
0       3750.0    MALE  
1       3800.0  FEMALE  
2       3250.0  FEMALE  
4       3450.0  FEMALE  
5       3650.0    MALE  


In [9]:
flow_logs["Is_Anomalous"]=(flow_logs["species"]!="Adelie").astype(int)

In [10]:
print(flow_logs.head())

  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   
1  Adelie  Torgersen            39.5           17.4              186.0   
2  Adelie  Torgersen            40.3           18.0              195.0   
4  Adelie  Torgersen            36.7           19.3              193.0   
5  Adelie  Torgersen            39.3           20.6              190.0   

   body_mass_g     sex  Is_Anomalous  
0       3750.0    MALE             0  
1       3800.0  FEMALE             0  
2       3250.0  FEMALE             0  
4       3450.0  FEMALE             0  
5       3650.0    MALE             0  


In [11]:

feature_columns = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

target_column = "Is_Anomalous"

In [12]:
network_logs = flow_logs[
    feature_columns + [target_column]
].copy()



In [13]:
print("\n[TASK 1 SUCCESS] API Data Fetched Successfully!")

print("\nProcessed Network Logs Shape:",
      network_logs.shape)

print("\nClass Distribution (0: Normal / 1: Anomalous):")

print(network_logs[target_column].value_counts().sort_index())
print("\nFirst 3 Rows:")

print(network_logs.head(3))


[TASK 1 SUCCESS] API Data Fetched Successfully!

Processed Network Logs Shape: (333, 5)

Class Distribution (0: Normal / 1: Anomalous):
Is_Anomalous
0    146
1    187
Name: count, dtype: int64

First 3 Rows:
   bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  Is_Anomalous
0            39.1           18.7              181.0       3750.0             0
1            39.5           17.4              186.0       3800.0             0
2            40.3           18.0              195.0       3250.0             0


In [14]:
X = network_logs[feature_columns]

y = network_logs[target_column]


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [16]:
scaler = StandardScaler()

# Learn mean/std ONLY from training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to test data
X_test_scaled = scaler.transform(X_test)

In [17]:

print("\n[TASK 2 SUCCESS] Data Partitioned & Scaled.")

print("\nTraining Features Shape :", X_train_scaled.shape)

print("Testing Features Shape  :", X_test_scaled.shape)


[TASK 2 SUCCESS] Data Partitioned & Scaled.

Training Features Shape : (233, 4)
Testing Features Shape  : (100, 4)


In [18]:
print("\nTrain Target Balance :",
      f"0 -> {(y_train == 0).sum()} | "
      f"1 -> {(y_train == 1).sum()}")


print("Test Target Balance  :",
      f"0 -> {(y_test == 0).sum()} | "
      f"1 -> {(y_test == 1).sum()}")


Train Target Balance : 0 -> 102 | 1 -> 131
Test Target Balance  : 0 -> 44 | 1 -> 56


In [19]:
model = LogisticRegression(
    random_state=42
)


# Train model
model.fit(
    X_train_scaled,
    y_train
)

LogisticRegression(random_state=42)

In [20]:
beta_0 = model.intercept_[0]

In [21]:
coefficients = model.coef_[0]


print("\n[TASK 3 SUCCESS] Logistic Regression Model Trained.")

print(
    f"\nModel Intercept (Beta_0): {beta_0:.4f}"
)

print("\nFeature Coefficients (Beta_i):")

for feature, coefficient in zip(
    feature_columns,
    coefficients
):
    print(
        f" - {feature:<20}: {coefficient: .4f}"
    )


[TASK 3 SUCCESS] Logistic Regression Model Trained.

Model Intercept (Beta_0): 0.9766

Feature Coefficients (Beta_i):
 - bill_length_mm      :  3.7992
 - bill_depth_mm       : -1.8720
 - flipper_length_mm   :  0.4445
 - body_mass_g         : -0.3959


In [22]:
probabilities = model.predict_proba(
    X_test_scaled
)


# Probability of anomalous class
anomalous_probabilities = probabilities[:, 1]


# ================================================================
# DEFAULT DECISION THRESHOLD = 0.50
# ================================================================

predicted_labels = (
    anomalous_probabilities >= 0.50
).astype(int)


print(
    "\n[TASK 4 SUCCESS] Probabilities Evaluated "
    "for Test Samples:"
)


[TASK 4 SUCCESS] Probabilities Evaluated for Test Samples:


In [23]:
print(y_test)

295    1
96     0
268    1
334    1
307    1
      ..
94     0
58     0
325    1
146    0
46     0
Name: Is_Anomalous, Length: 100, dtype: int64


In [24]:

for i in range(5):

    probability = anomalous_probabilities[i]

    actual_label = y_test.iloc[i]

    if probability > 0.50:
        risk_level = "High Risk"
    else:
        risk_level = "Low Risk"

    print(
        f"Sample {i + 1} | "
        f"P(Anomalous): {probability:.4f} | "
        f"Label: {actual_label} | "
        f"Assigned: {risk_level}"
    )

Sample 1 | P(Anomalous): 0.9962 | Label: 1 | Assigned: High Risk
Sample 2 | P(Anomalous): 0.0117 | Label: 0 | Assigned: Low Risk
Sample 3 | P(Anomalous): 0.9953 | Label: 1 | Assigned: High Risk
Sample 4 | P(Anomalous): 0.9975 | Label: 1 | Assigned: High Risk
Sample 5 | P(Anomalous): 0.9999 | Label: 1 | Assigned: High Risk


In [25]:
thresholds = [0.30, 0.50, 0.70]


print(
    "\n========== DECISION BOUNDARY SENSITIVITY ANALYSIS =========="
)


for threshold in thresholds:
    custom_predictions = (
        anomalous_probabilities >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_test,
        custom_predictions
    )

    precision = precision_score(
        y_test,
        custom_predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        custom_predictions,
        zero_division=0
    )

    print(
        f"\nThreshold: {threshold:.2f} | "
        f"Accuracy: {accuracy * 100:.1f}% | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f}"
    )




========== DECISION BOUNDARY SENSITIVITY ANALYSIS ==========

Threshold: 0.30 | Accuracy: 99.0% | Precision: 0.9825 | Recall: 1.0000

Threshold: 0.50 | Accuracy: 100.0% | Precision: 1.0000 | Recall: 1.0000

Threshold: 0.70 | Accuracy: 100.0% | Precision: 1.0000 | Recall: 1.0000


In [26]:
# ================================================================
# FINAL EXPECTED OUTPUT / SUMMARY
# ================================================================

base_predictions = (
    anomalous_probabilities >= 0.50
).astype(int)

base_accuracy = accuracy_score(
    y_test,
    base_predictions
)


print(
    "\n========== API-DRIVEN LOGISTIC REGRESSION "
    "& DECISION BOUNDARY ENGINE =========="
)

print(
    "\nData Ingestion Status      : "
    "REST API Ingestion Successful (HTTP 200 OK)"
)

print(
    f"Master Dataset Records     : {len(network_logs)} "
    "(Cleaned & Preprocessed)"
)

print(
    "Features Included          : "
    "4 Continuous Log Metrics "
    "(bill_length_mm, bill_depth_mm, "
    "flipper_length_mm, body_mass_g)"
)

print(
    "Target Output              : "
    "Is_Anomalous "
    "(Binary Classification: 0 = Standard, 1 = Anomalous)"
)


print("\nModel Training Metrics:")

print(
    f"- Stratified Train Split   : {len(X_train)} Records "
    f"({(y_train == 0).sum()} Normal / "
    f"{(y_train == 1).sum()} Anomalous)"
)

print(
    f"- Stratified Test Split    : {len(X_test)} Records "
    f"({(y_test == 0).sum()} Normal / "
    f"{(y_test == 1).sum()} Anomalous)"
)

print(
    f"- Base Test Accuracy       : "
    f"{base_accuracy * 100:.1f}% "
    "(at Default 0.50 Threshold)"
)


print("\nSigmoid Probability Profiling:")

print(
    f"- Maximum Anomalous Prob   : "
    f"{anomalous_probabilities.max() * 100:.2f}%"
)

print(
    f"- Minimum Anomalous Prob   : "
    f"{anomalous_probabilities.min() * 100:.2f}%"
)


print(
    f"- Key Probability Drivers  : "
    f"{feature_columns[0]} "
    f"({coefficients[0]:+.4f}), "
    f"{feature_columns[1]} "
    f"({coefficients[1]:+.4f})"
)


print("\nDecision Threshold Tuning:")

for threshold in thresholds:

    predictions = (
        anomalous_probabilities >= threshold
    ).astype(int)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    print(
        f"- Threshold @ {threshold:.2f} : "
        f"{accuracy * 100:.1f}% Accuracy | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f}"
    )


print(
    "\nConclusion:"
)

print(
    "By implementing dynamic REST API fetching, the data "
    "pipeline dynamically ingests raw data over HTTP. "
    "Logistic Regression maps continuous features to "
    "sigmoid probability scores, enabling the decision "
    "threshold to be adjusted according to the desired "
    "classification behavior."
)


========== API-DRIVEN LOGISTIC REGRESSION & DECISION BOUNDARY ENGINE ==========

Data Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records     : 333 (Cleaned & Preprocessed)
Features Included          : 4 Continuous Log Metrics (bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)
Target Output              : Is_Anomalous (Binary Classification: 0 = Standard, 1 = Anomalous)

Model Training Metrics:
- Stratified Train Split   : 233 Records (102 Normal / 131 Anomalous)
- Stratified Test Split    : 100 Records (44 Normal / 56 Anomalous)
- Base Test Accuracy       : 100.0% (at Default 0.50 Threshold)

Sigmoid Probability Profiling:
- Maximum Anomalous Prob   : 100.00%
- Minimum Anomalous Prob   : 0.11%
- Key Probability Drivers  : bill_length_mm (+3.7992), bill_depth_mm (-1.8720)

Decision Threshold Tuning:
- Threshold @ 0.30 : 99.0% Accuracy | Precision: 0.9825 | Recall: 1.0000
- Threshold @ 0.50 : 100.0% Accuracy | Precision: 1.0000 | Reca